Chuẩn bị dữ liệu

In [6]:
from tensorflow import keras
# ImageDataGenerator là 1 layer của Kerá giúp tiền xử lý ảnh
from tensorflow.keras.preprocessing.image import ImageDataGenerator 

In [7]:
data_dir = "./asl-alphabet/asl_alphabet_train/asl_alphabet_train"
input_size = (128, 128) # Đặt lại kích thước tất cả anh về cùng 1 kích thước 128x128
val_frac = 0.1          # Tỉ lệ chia dữ liệu, trong đó 10% dữ liệu sẽ được sử dụng cho tập validation (xác thực), và phần còn lại sẽ được dùng cho tập huấn luyện.
batch_size = 128        # Số lượng ảnh được xử lý trong mỗi lần huấn luyện

# Tạo 1 đối tượng để chuẩn hóa ảnh trước khi đưa vào mô hình
data_augmentor = ImageDataGenerator(samplewise_center=True,             # Tất cả giá trị pixel của ảnh sẽ được điều chỉnh sao cho trung bình của các pixel là 0.
                                    samplewise_std_normalization=True,  # Tất cả giá trị pixel sẽ được chuẩn hóa sao cho độ lệch chuẩn (standard deviation) của các pixel là 1.
                                    validation_split=val_frac           # Đảm bảo rằng 10% dữ liệu được chia làm tập validation.
                                    )

# Tạo 1 generator để huấn luyện
train_generator = data_augmentor.flow_from_directory(data_dir,
                                                     target_size = input_size,  # Thay đổi kích thước ảnh thành (128, 128).
                                                     batch_size = batch_size,   # Số ảnh trong mỗi batch.
                                                     shuffle=True,              # Xáo trộn các ảnh trong mỗi batch để tăng tính ngẫu nhiên và giúp mô hình học tốt hơn.
                                                     subset="training"          # Chỉ định đây là tập huấn luyện.
                                                     )

# Tạo 1 generator để xác thực
val_generator = data_augmentor.flow_from_directory(data_dir,
                                                     target_size = input_size,
                                                     batch_size = batch_size,
                                                     subset="validation")

Found 78300 images belonging to 29 classes.
Found 8700 images belonging to 29 classes.


In [8]:
# 29 classes tương ứng với 29 folders là 29 ký tự
# Trả về dictionary ánh xạ giữa tên class (A, B, C,...) và chỉ số của class đó
train_generator.class_indices

{'A': 0,
 'B': 1,
 'C': 2,
 'D': 3,
 'E': 4,
 'F': 5,
 'G': 6,
 'H': 7,
 'I': 8,
 'J': 9,
 'K': 10,
 'L': 11,
 'M': 12,
 'N': 13,
 'O': 14,
 'P': 15,
 'Q': 16,
 'R': 17,
 'S': 18,
 'T': 19,
 'U': 20,
 'V': 21,
 'W': 22,
 'X': 23,
 'Y': 24,
 'Z': 25,
 'del': 26,
 'nothing': 27,
 'space': 28}

Xây dựng mô hình

In [ ]:
num_classes = len(train_generator.class_indices)
input_shape = (128, 128, 3) # Kích thức của ảnh đầu vào là 128x128 pixel, 3 kênh màu (RGB)

# Xây dựng mô hình học sâu (CNN)
model = keras.models.Sequential([
    keras.Input(shape=input_shape),
    # block 1
    keras.layers.Conv2D(32, kernel_size=(3, 3), activation='relu'), # Layer tích chập
    keras.layers.Conv2D(32, kernel_size=(3, 3), activation='relu'), # Thêm Layer tích chập để học nhiều chi tiết hơn
    keras.layers.MaxPooling2D(pool_size=(2, 2)),                    # Layer MaxPool luôn theo sau layer Conv, Lớp max-pooling giúp giảm kích thước ảnh, giúp mô hình học nhanh và ít tốn tài nguyên.
    keras.layers.Dropout(0.5),                                      

    # block 2
    keras.layers.Conv2D(64, kernel_size=(3, 3), activation='relu'),
    keras.layers.Conv2D(64, kernel_size=(3, 3), activation='relu'),
    keras.layers.MaxPooling2D(pool_size=(2, 2)),
    keras.layers.Dropout(0.5),

    # block 3
    keras.layers.Conv2D(128, kernel_size=(3, 3), activation='relu'),
    keras.layers.Conv2D(128, kernel_size=(3, 3), activation='relu'),
    keras.layers.MaxPooling2D(pool_size=(2, 2)),
    keras.layers.Dropout(0.5),

    # FCN block chịu trách nhiệm phân loại
    keras.layers.Flatten(),                                          # Chỉ có một layer flatten trong mô hình, Làm phẳng ảnh từ dạng ma trận (2D) thành một vector (1D) để đưa vào các lớp fully-connected (FC).
    keras.layers.Dense(512, activation='relu'),
    keras.layers.Dense(128, activation='relu'),
    keras.layers.Dropout(0.5),
    keras.layers.Dense(num_classes, activation='softmax')           # Layer Dense cuối có chiều là tổng số class, Các lớp fully connected, nơi mô hình học các mối quan hệ giữa các đặc trưng đã trích xuất từ ảnh.
])

model.compile(optimizer='adam',                 # Thuật toán tối ưu Adam giúp mô hình học nhanh và hiệu quả.
              loss='categorical_crossentropy',  # Hàm mất mát dùng cho bài toán phân loại nhiều lớp.
              metrics=["accuracy"])             # Đánh giá mô hình dựa trên độ chính xác
model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_8 (Conv2D)               │ (None, 126, 126, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_9 (Conv2D)               │ (None, 124, 124, 32)   │         9,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_4 (MaxPooling2D)  │ (None, 62, 62, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ (None, 62, 62, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_10 (Conv2D)              │ (None, 60, 60, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_11 (Conv2D)              │ (None, 58, 58, 64)     │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_5 (MaxPooling2D)  │ (None, 29, 29, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_5 (Dropout)             │ (None, 29, 29, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_12 (Conv2D)              │ (None, 27, 27, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_13 (Conv2D)              │ (None, 25, 25, 128)    │       147,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_6 (MaxPooling2D)  │ (None, 12, 12, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_6 (Dropout)             │ (None, 12, 12, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ (None, 18432)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 512)            │     9,437,696 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 128)            │        65,664 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_7 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 29)             │         3,741 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 9,794,109 (37.36 MB)

 Trainable params: 9,794,109 (37.36 MB)

 Non-trainable params: 0 (0.00 B)

Huấn luyện mô hình

In [9]:
# Bắt đầu huấn luyện mô hình với dữ liệu từ train_generator trong 5 vòng lặp
model.fit(train_generator,
          epochs=5,
          validation_data=val_generator)

Epoch 1/5
612/612 ━━━━━━━━━━━━━━━━━━━━ 972s 2s/step - accuracy: 0.5225 - loss: 1.5653 - val_accuracy: 0.7754 - val_loss: 0.6241
Epoch 2/5
612/612 ━━━━━━━━━━━━━━━━━━━━ 765s 1s/step - accuracy: 0.8945 - loss: 0.3091 - val_accuracy: 0.8336 - val_loss: 0.5578
Epoch 3/5
612/612 ━━━━━━━━━━━━━━━━━━━━ 1068s 2s/step - accuracy: 0.9470 - loss: 0.1623 - val_accuracy: 0.8397 - val_loss: 0.6111
Epoch 4/5
612/612 ━━━━━━━━━━━━━━━━━━━━ 1456s 2s/step - accuracy: 0.9654 - loss: 0.1059 - val_accuracy: 0.8576 - val_loss: 0.5425
Epoch 5/5
612/612 ━━━━━━━━━━━━━━━━━━━━ 10601s 17s/step - accuracy: 0.9742 - loss: 0.0803 - val_accuracy: 0.8567 - val_loss: 0.5695


In [ ]:
# Lưu model để lần sau nó sẽ sử dụng luôn kết quả đã huấn luyện
model.save('model.keras')

Dự đoán mô hình

In [19]:
from tensorflow.keras.saving import load_model
loaded_model = load_model("./model.keras")

In [12]:
import matplotlib.pyplot as plt # Hiển thị ảnh ra màn hình
import cv2 # Thư viện xử lý ảnh chuyên nghiệp
from glob import glob # Lấy danh sách file
import numpy as np

In [4]:
# Sử dùng hàm glob để đọc ảnh từ thư mục test
test_images = glob("./asl-alphabet/asl_alphabet_test/asl_alphabet_test/**")
test_images[:5]

['./asl-alphabet/asl_alphabet_test/asl_alphabet_test\\A_test.jpg',
 './asl-alphabet/asl_alphabet_test/asl_alphabet_test\\B_test.jpg',
 './asl-alphabet/asl_alphabet_test/asl_alphabet_test\\C_test.jpg',
 './asl-alphabet/asl_alphabet_test/asl_alphabet_test\\D_test.jpg',
 './asl-alphabet/asl_alphabet_test/asl_alphabet_test\\E_test.jpg']

In [9]:
# Tạo dictionary chứa index tương ứng với tên class có dạng {'A': 0, 'B': 1, 'C': 2}
classes = train_generator.class_indices
classes = dict((v, k) for k, v in classes.items()) # đảo ngược dictionary {0: 'A', 1: 'B', 2: 'C'}
classes


{0: 'A',
 1: 'B',
 2: 'C',
 3: 'D',
 4: 'E',
 5: 'F',
 6: 'G',
 7: 'H',
 8: 'I',
 9: 'J',
 10: 'K',
 11: 'L',
 12: 'M',
 13: 'N',
 14: 'O',
 15: 'P',
 16: 'Q',
 17: 'R',
 18: 'S',
 19: 'T',
 20: 'U',
 21: 'V',
 22: 'W',
 23: 'X',
 24: 'Y',
 25: 'Z',
 26: 'del',
 27: 'nothing',
 28: 'space'}

In [ ]:
# Lấy ra ảnh để test
test_image_path = test_images[3]
im = cv2.imread(test_image_path) # Đọc ảnh từ ổ cứng
plt.rcParams['figure.figsize'] = (2.0, 2.0) # Chỉnh kích thước ảnh đơn vị inch
plt.imshow(cv2.cvtColor(im, cv2.COLOR_BGR2RGB)) # Chuyển mã màu ảnh từ BRG -> RGB
plt.title(test_image_path.split("/")[-1]) # Lấy tên file, hiển thị lên tiêu đề

Text(0.5, 1.0, 'asl_alphabet_test\\D_test.jpg')

In [21]:
from tensorflow.keras.preprocessing import image # Module xử lý ảnh của Keras
img = image.load_img(test_image_path, target_size=(input_size)) # Resize về 128 x 128
img_array = image.img_to_array(img) # Chuyển kiểu dữ liệu từ hình sang kiểu numpy array

img_array = np.expand_dims(img_array, axis=0) # Thêm chiều n=1 để dự đoán
test_datagen = ImageDataGenerator(
    samplewise_center = True,
    samplewise_std_normalization = True
)
img_generator = test_datagen.flow(img_array, batch_size=1)

prediction = loaded_model.predict(next(img_generator))
prediction_label = np.argmax(prediction)
print("Prediction:", classes[prediction_label])

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 334ms/step
Prediction: D
